In [1]:
# 07b_governance_llm_ablation.ipynb
# =============================================================================
# Notebook 07b — Governance Compliance with LLM-only / HardRule-only ablation
#
# Revision (Reviewer A #4 & #5). Evaluates G1-G4 compliance for FIVE conditions
# at the individual CF-candidate level:
#     PreGuardrail   : pure DiCE, no constraints
#     LLM_strict     : DiCE constrained to LLM raw ranges (strict parse)
#     LLM_lenient    : DiCE constrained to LLM raw ranges (lenient parse)
#     HardRule       : DiCE constrained to Hard-Rule-only ranges (no LLM)
#     Final          : DiCE constrained to LLM+HardRule ranges
#
# For a feature the LLM did NOT supply (LLM-only conditions), that feature is
# left UNCONSTRAINED in the DiCE search (permitted_range omits it), so the
# resulting CF reflects what the LLM's guardrail would actually have governed.
#
# Input  : ../results/tables/agent_config.pkl, df_final.pkl, model_{group}.pkl,
#          ../results/tables/guardrail_ranges_v2.json
# Output : ../results/tables/governance_llm_ablation.csv
#          ../results/tables/governance_llm_ablation_stats.csv
#          ../results/tables/table4_llmonly.csv   (paper Table 4)
#          ../results/tables/table5_llmonly_bycase.csv
#          ../results/tables/table6_llmonly_bystratum.csv
# =============================================================================

# %%
import json, joblib, warnings
import numpy as np, pandas as pd
import dice_ml
from statsmodels.stats.contingency_tables import mcnemar
from scipy.stats import fisher_exact

warnings.filterwarnings('ignore')

agent_config = joblib.load('../results/tables/agent_config.pkl')
df_final     = joblib.load('../results/tables/df_final.pkl')
X_FEATURES      = agent_config['X_features']
VARY_FEATURES   = agent_config['vary_features']
TARGET_COL      = agent_config['target_col']
AGEGROUP_CONFIG = agent_config['agegroup_config']

models = {g: joblib.load(f'../results/tables/model_{g}.pkl')
          for g in AGEGROUP_CONFIG}

with open('../results/tables/guardrail_ranges_v2.json', encoding='utf-8') as f:
    GR = json.load(f)

# ---- G1-G4 checkers (identical logic to nb07) --------------------------------
def g1_physical_consistency(cf, orig, tol=1.0):
    def sign(k):
        o = float(orig.get(k, 0)); v = float(cf.get(k, o))
        pct = (v - o)/o*100 if o != 0 else 0
        return np.sign(pct) if abs(pct) > tol else 0
    s = [sign(k) for k in ['BMI', 'WaistCirc', 'Weight']]
    nz = [x for x in s if x != 0]
    return True if not nz else len(set(nz)) == 1

def g2_energy_safety(cf, orig):
    c = float(orig.get('Energy_kcal', 0)); v = float(cf.get('Energy_kcal', c))
    return max(c*0.70, 500.0) <= v <= c*1.01

def g3_conflict_prevention(cf, orig, tol=1.0):
    c_na = float(orig.get('Sodium_mg', 0)); v_na = float(cf.get('Sodium_mg', c_na))
    if not (c_na > 0 and (c_na - v_na)/c_na*100 > tol):
        return True
    c_c = float(orig.get('Carb_g', 0));  v_c = float(cf.get('Carb_g', c_c))
    c_s = float(orig.get('Sugar_g', 0)); v_s = float(cf.get('Sugar_g', c_s))
    carb_ok  = (v_c - c_c)/c_c*100 <= tol if c_c > 0 else True
    sugar_ok = (v_s - c_s)/c_s*100 <= tol if c_s > 0 else True
    return carb_ok and sugar_ok

def g4_macronutrient_floors(cf, orig):
    for feat, ratio, amin in [('Protein_g',0.60,30.0),('Potassium_mg',0.50,500.0),
                              ('Carb_g',0.20,30.0),('Fiber_g',0.30,5.0)]:
        c = float(orig.get(feat, 0)); v = float(cf.get(feat, c))
        if v < max(c*ratio, amin)*0.99:
            return False
    return True

def build_search_range(ranges, df_ref, features, only_these=None):
    """permitted_range for DiCE. If only_these is given, restrict to those
    features (LLM-only: unconstrained features are omitted)."""
    sr = {}
    feats = features if only_these is None else [f for f in features if f in only_these]
    for feat in feats:
        lo, hi = ranges.get(feat, [None, None])
        if lo is None:
            continue
        data_lo = float(df_ref[feat].quantile(0.05))
        data_hi = float(df_ref[feat].quantile(0.95))
        sr[feat] = [round(max(min(lo, data_lo), 0.0), 4),
                    round(max(hi, data_hi), 4)]
    return sr

def gen_cfs(exp, query, permitted=None, n=4):
    kw = dict(total_CFs=n, desired_class=0, features_to_vary=VARY_FEATURES,
              proximity_weight=0.2, sparsity_weight=0.1)
    if permitted is not None:
        kw['permitted_range'] = permitted
    try:
        cf = exp.generate_counterfactuals(query, **kw)
        cfs = cf.cf_examples_list[0].final_cfs_df.to_dict('records') if cf else []
    except Exception:
        cfs = []
    # hard-clip to permitted (mirrors nb06 behaviour)
    if permitted:
        for r in cfs:
            for feat, (lo, hi) in permitted.items():
                if feat in r:
                    r[feat] = float(np.clip(float(r[feat]), lo, hi))
    return cfs

# %%
records = []
for case_key, g in GR.items():
    grp   = g['group']; mdl = models.get(grp)
    if mdl is None:
        continue
    orig  = g['patient_profile']
    cfg   = AGEGROUP_CONFIG[grp]
    age   = 0.0 if cfg['age_min'] < 60 else 1.0
    dref  = df_final[(df_final['AgeGroup']==age) & (df_final['Sex']==cfg['sex_code'])].copy()
    query = pd.DataFrame([orig])[X_FEATURES]

    d = dice_ml.Data(dataframe=df_final.copy().astype(float)[X_FEATURES+[TARGET_COL]],
                     continuous_features=X_FEATURES, outcome_name=TARGET_COL)
    m = dice_ml.Model(model=mdl, backend='sklearn')
    exp = dice_ml.Dice(d, m, method='genetic')

    conditions = {
        'PreGuardrail': None,
        'LLM_strict':   build_search_range(g['llm_raw_strict'],  dref, X_FEATURES,
                                            only_these=set(g['llm_raw_strict'])),
        'LLM_lenient':  build_search_range(g['llm_raw_lenient'], dref, X_FEATURES,
                                            only_these=set(g['llm_raw_lenient'])),
        'HardRule':     build_search_range(g['hardrule_ranges'], dref, X_FEATURES),
        'Final':        build_search_range(g['final_ranges'],    dref, X_FEATURES),
    }

    for cond, permitted in conditions.items():
        cfs = gen_cfs(exp, query, permitted)
        for i, cf in enumerate(cfs, 1):
            records.append({
                'CaseKey': case_key, 'Group': grp, 'Condition': cond, 'CF_idx': i,
                'G1': int(g1_physical_consistency(cf, orig)),
                'G2': int(g2_energy_safety(cf, orig)),
                'G3': int(g3_conflict_prevention(cf, orig)),
                'G4': int(g4_macronutrient_floors(cf, orig)),
            })
    print(f"  [{case_key}] done")

gov = pd.DataFrame(records)
gov.to_csv('../results/tables/governance_llm_ablation.csv',
           index=False, encoding='utf-8-sig')

# %%
# ---- Table 4: compliance rate by condition × dimension -----------------------
rate = (gov.groupby('Condition')[['G1','G2','G3','G4']].mean()*100).round(1)
order = ['PreGuardrail','LLM_strict','LLM_lenient','HardRule','Final']
rate = rate.reindex([c for c in order if c in rate.index])
rate.to_csv('../results/tables/table4_llmonly.csv', encoding='utf-8-sig')
print("\n=== Table 4 (compliance % by condition) ===")
print(rate.to_string())

# ---- Table 5: LLM-only pass/fail by case (strict) ----------------------------
def case_pass(sub):
    # a case 'passes' a dimension if ALL its CF candidates pass
    return {d: int(sub[d].min()) for d in ['G1','G2','G3','G4']}
t5 = []
for ck in GR:
    sub = gov[(gov['CaseKey']==ck) & (gov['Condition']=='LLM_strict')]
    row = {'CaseKey': ck, 'Group': GR[ck]['group'],
           'LLM_n_supplied': GR[ck]['llm_n_strict']}
    if len(sub):
        row.update(case_pass(sub))
    else:
        row.update({d: np.nan for d in ['G1','G2','G3','G4']})
    t5.append(row)
t5 = pd.DataFrame(t5)
t5.to_csv('../results/tables/table5_llmonly_bycase.csv',
          index=False, encoding='utf-8-sig')
print("\n=== Table 5 (LLM-only strict, pass=1/fail=0 by case) ===")
print(t5.to_string(index=False))

# ---- Table 6: LLM-only pass rate by stratum (strict) -------------------------
t6 = (gov[gov['Condition']=='LLM_strict']
      .groupby('Group')[['G1','G2','G3','G4']].mean()*100).round(1)
t6.to_csv('../results/tables/table6_llmonly_bystratum.csv', encoding='utf-8-sig')
print("\n=== Table 6 (LLM-only strict, pass rate % by stratum) ===")
print(t6.to_string())

# %%
# ---- McNemar: Pre vs Final, and HardRule vs Final ---------------------------
def mcnemar_pair(cond_a, cond_b):
    a = gov[gov['Condition']==cond_a].set_index(['CaseKey','CF_idx'])
    b = gov[gov['Condition']==cond_b].set_index(['CaseKey','CF_idx'])
    idx = a.index.intersection(b.index)
    out = []
    for dim in ['G1','G2','G3','G4']:
        av, bv = a.loc[idx, dim].values, b.loc[idx, dim].values
        n01 = int(((av==0)&(bv==1)).sum()); n10 = int(((av==1)&(bv==0)).sum())
        n11 = int(((av==1)&(bv==1)).sum()); n00 = int(((av==0)&(bv==0)).sum())
        try:
            p = mcnemar(np.array([[n11,n10],[n01,n00]]), exact=True).pvalue
        except Exception:
            p = np.nan
        out.append({'Comparison': f'{cond_a}_vs_{cond_b}', 'Dim': dim,
                    'A_pct': round(av.mean()*100,1), 'B_pct': round(bv.mean()*100,1),
                    'Delta': round((bv.mean()-av.mean())*100,1),
                    'p_value': round(p,6) if not np.isnan(p) else np.nan})
    return out

stats = []
for a, b in [('PreGuardrail','Final'), ('HardRule','Final'),
             ('LLM_strict','Final'),   ('PreGuardrail','HardRule')]:
    stats += mcnemar_pair(a, b)
stats_df = pd.DataFrame(stats)
stats_df.to_csv('../results/tables/governance_llm_ablation_stats.csv',
                index=False, encoding='utf-8-sig')
print("\n=== McNemar comparisons ===")
print(stats_df.to_string(index=False))

100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.37it/s]


  [MiddleAged_Male_case1] done


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.34it/s]


  [MiddleAged_Male_case2] done


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.52it/s]


...

  [Older_Female_case1] done


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.92it/s]


  [Older_Female_case2] done


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.88it/s]

  [Older_Female_case3] done

=== Table 4 (compliance % by condition) ===
                G1    G2    G3    G4
Condition                           
PreGuardrail  91.7  39.6  68.8  75.0
LLM_strict    91.5  44.7  66.0  72.3
LLM_lenient   91.5  38.3  63.8  61.7
HardRule      85.1  44.7  61.7  85.1
Final         91.3  45.7  54.3  78.3

=== Table 5 (LLM-only strict, pass=1/fail=0 by case) ===
                CaseKey             Group  LLM_n_supplied  G1  G2  G3  G4
  MiddleAged_Male_case1   MiddleAged_Male              31   1   0   0   0
  MiddleAged_Male_case2   MiddleAged_Male              31   1   0   0   0
  MiddleAged_Male_case3   MiddleAged_Male              31   0   0   0   1
MiddleAged_Female_case1 MiddleAged_Female              31   0   0   0   0
MiddleAged_Female_case2 MiddleAged_Female              31   1   0   1   1
MiddleAged_Female_case3 MiddleAged_Female              31   0   0   1   0
       Older_Male_case1        Older_Male              31   1   1   0   0
       Older_Male_